# EPA Model — XGBoost Regression

Trains an XGBoost regressor to predict EPA (Expected Points Added) per play  
from game-state features. Ground truth EPA labels come from nflfastR.  
Validation is time-based: train on 2021–2022, validate on 2023.  

After running this notebook, call `json_exporter.py` to push results to the web page.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import shap
from pathlib import Path

pio.renderers.default = 'notebook_connected'

import sys
sys.path.insert(0, str(Path('../../').resolve()))

from python.features.engineering import add_features, get_feature_cols, time_split
from python.features.selection import top_features
from python.models.epa_model import EPAModel

DATA = Path('../../data/outputs/play_features.csv')
print(f'loading data from {DATA}')

In [ ]:
df = pd.read_csv(DATA)
print(f'  raw rows: {len(df):,}')

# drop NA rows
df = df.dropna(subset=['epa', 'down', 'ydstogo', 'yardline_100'])
print(f'  after dropna: {len(df):,}')
print(f'  seasons: {sorted(df["season"].unique())}')
print(f'  plays by season:\n{df.groupby("season").size()}')

In [ ]:
# feature engineering
df_f = add_features(df)
feat = get_feature_cols()
feat = [c for c in feat if c in df_f.columns]
print(f'using {len(feat)} features: {feat}')

In [ ]:
# time-based split: 2021-2022 train, 2023 val
train, val = time_split(df_f, val_season=2023)
X_tr, y_tr = train[feat], train['epa']
X_v,  y_v  = val[feat],   val['epa']
print(f'train: {len(X_tr):,} plays | val: {len(X_v):,} plays')

# EPA distribution
fig = go.Figure()
fig.add_trace(go.Histogram(x=y_tr, name='train', opacity=0.6, nbinsx=100))
fig.add_trace(go.Histogram(x=y_v,  name='val',   opacity=0.6, nbinsx=100))
fig.update_layout(title='EPA Distribution — Train vs Val', barmode='overlay',
                  xaxis_title='EPA', yaxis_title='count', template='simple_white')
fig.show()

In [ ]:
# train model with early stopping
model = EPAModel()
model.fit_with_eval(X_tr, y_tr, X_v, y_v)
print('training complete')

In [ ]:
# evaluation metrics
tr_metrics = model.evaluate(X_tr, y_tr)
v_metrics  = model.evaluate(X_v,  y_v)
print(f'train: {tr_metrics}')
print(f'val:   {v_metrics}')

# note: EPA is inherently noisy (~0.8 RMSE is typical)
# nflfastR's EP model achieves ~0.82 RMSE; we should be in the same range

In [ ]:
# predicted vs actual (val set)
preds = model.predict(X_v)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_v[:3000], y=preds[:3000],
    mode='markers', marker=dict(size=2, opacity=0.3, color='#2271b3')
))
fig.add_shape(type='line', x0=y_v.min(), y0=y_v.min(),
              x1=y_v.max(), y1=y_v.max(),
              line=dict(dash='dash', color='red'))
fig.update_layout(title='Predicted vs Actual EPA (val 2023, first 3k plays)',
                  xaxis_title='actual EPA', yaxis_title='predicted EPA',
                  template='simple_white')
fig.show()

In [ ]:
# feature importance — gain
fi_df = top_features(model.model, feat, n=20)

fig = go.Figure(go.Bar(
    x=fi_df['importance'], y=fi_df['feature'],
    orientation='h', marker_color='#2271b3'
))
fig.update_layout(title='Feature Importance (gain)',
                  xaxis_title='importance', yaxis={'categoryorder': 'total ascending'},
                  template='simple_white', height=500)
fig.show()

In [ ]:
# SHAP values — sample 5000 plays for speed
sample = X_v.sample(5000, random_state=42)
explainer = shap.TreeExplainer(model.model)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type='bar', max_display=20)

In [ ]:
# team-level residuals: model-derived offensive efficiency
# residual > 0 means the team outperforms expectation given game state
val2 = val.copy()
val2['pred_epa'] = preds
val2['residual'] = val2['epa'] - val2['pred_epa']

team_residuals = (
    val2.groupby('posteam')
        .agg(mean_residual=('residual', 'mean'), plays=('residual', 'count'))
        .sort_values('mean_residual', ascending=False)
)
print('2023 team EPA residuals (top over-performers):')
print(team_residuals.head(10).round(4))

In [ ]:
# save model
saved_path = model.save()
print(f'model saved to {saved_path}')
print('next: run notebooks/models/02_wp_model.ipynb')